In [1]:
import numpy as np
import sys, os
import autograd.numpy as np
import matplotlib.pyplot as plt
import io 

In [2]:
def get_rp(m1, m2, b, v):

    R_sun = 7e8 #m
    M_sun = 2e30 #kg
    G = 6.67e-11 #Nm2/kg2

    #Convert units of masses (Msun), impact parameter (Rsun), and velocity (km/s)
    m1 *= M_sun
    m2 *= M_sun
    b *= R_sun
    v *= 10**(3)
    
    if b == 0.0:
        return 0.0
    else:
        A = (G * (m1 + m2))/(v**2 * b)

        result = b * (np.sqrt(1 + A**2) - A) #in meters 

        return result/R_sun

In [3]:
def get_star_id(collid):

        Star_id1 = Star_id1_all[Coll_id_cond == collid]
        Star_id2 = Star_id2_all[Coll_id_cond == collid]    
              
        return Star_id1, Star_id2


def get_star_radius(Star_id1, Star_id2):

        Star_rad1 = Star_rad_all[Star_id_all == Star_id1]
        Star_rad2 = Star_rad_all[Star_id_all == Star_id2]

        return Star_rad1, Star_rad2

**Collaborator's Data**

In [ ]:
directory = './data' 

bs= []
vrels = []
labels = []
dms = []
mass1s = []
mass_ratios = []

# List all files in the directory
for filename in os.listdir(directory):
    if filename != 'summary.txt':
        print("Looking at file", filename)
        file_path = os.path.join(directory, filename)
        data = np.genfromtxt(file_path,skip_header = True, dtype = float)
        v_inf1 = data[0,24]
        v_inf2 = data[0,25]
        m_unbound = data[-1,18]

        #initial masses 
        mass2_0 = data[0,2] 
        mass1_0 = data[0,1]

        vrel = (v_inf1+v_inf2)*437.

        b = np.abs(data[0,23])

        mass2 = data[-1,2]
        mass1 = data[-1,1]

        #calculate fractional mass loss 
        # dm = (m_unbound)/(mass1_0 + mass2_0)
        
        #create my labels
        # 0 complete disruption of both stars
        # 1 merged 
        # 2 unmerged 


        if mass2 == 0.0 and mass1 != 0.0:
            outcome = 1
        elif mass2 != 0.0 and mass1 != 0.0:
            outcome = 2
        elif mass2 == 0.0 and mass1 == 0.0:
            outcome = 0
        else:
            print("IDK WHAT HAPPENED HERE")
            

        bs.append(b)
        vrels.append(vrel)
        labels.append(outcome)
        mass1s.append(mass1_0)
        mass_ratios.append(mass1_0/mass2_0)
        # dms.append(dm)

**Freitag and Benz Data**

In [8]:
#Code was adapted by code sent by Jamie 

bs= []
vrels = []
labels = []
dms = []
mass1s = []
mass2s = []
mass_ratios = []
rps = []

# Read and process summary.txt
with open('./data/summary.txt', 'r') as file:
    lines = file.readlines()

# Filter out lines that are just dashes
lines = [line for line in lines if not line.strip().startswith('-')]

# Convert the filtered lines to a single string and use genfromtxt
data = np.genfromtxt(io.StringIO(''.join(lines)), skip_header=52, names=True, dtype=None, encoding=None)

# Check the fields
print(data.dtype.names)

R_sun = 7e8
M_sun = 2e30
G = 6.67e-11
vunit = (G*M_sun/R_sun)**0.5 * 1e-3
print('velocity unit=',vunit,'km/s')

coll_id = data['Coll_id']
mini1 = data['Mini1']
mini2 = data['Mini2']
vrel_inf = data['Vrel_inf'] * vunit
imp_param = data['ImpParam']
mfin1 = data['Mfin1']
mfin2 = data['Mfin2']
n_star = data['Nstar']
dm = data["dM"]

# Look for places in data file where final star indices should be swapped
for i in range(len(coll_id)):
    if mfin1[i] > mfin2[i]:
        # print(f"Will swap Mfin1 and Mfin2 for Coll_id: {coll_id[i]}, Mini1: {mini1[i]}, Mini2: {mini2[i]}, Mfin1: {mfin1[i]}, M2: {mfin2[i]}")
        mfin1[i], mfin2[i] = mfin2[i], mfin1[i]

# vrel_inf = vrel_inf[(mini1 == mini2) & (mini1 == 1.)]
# b        = imp_param[(mini1 == mini2) & (mini1 == 1.)]
# mfin1    = mfin1[(mini1 == mini2) & (mini1 == 1.)]
# mfin2    = mfin2[(mini1 == mini2) & (mini1 == 1.)]
# dm       = dm[(mini1 == mini2) & (mini1 == 1.)]
# coll_id = coll_id[(mini1 == mini2) & (mini1 == 1.)]

print(len(mfin2))

# Read init_cond.txt **once**
with open('./FB_Dataset/init_cond.txt', 'r') as file:
    lines = [line for line in file if not line.strip().startswith('-')]

data_coll = np.genfromtxt(io.StringIO(''.join(lines)), skip_header=48, names=True, dtype=None, encoding=None)

Coll_id_cond = data_coll["Coll_id"]
Star_id1_all = data_coll["Star_id_1"].astype(int)
Star_id2_all = data_coll["Star_id_2"].astype(int)

# Read stars.txt **once**
with open('./FB_Dataset/stars.txt', 'r') as file:
    lines = [line for line in file if not line.strip().startswith('-')]

data_stars = np.genfromtxt(io.StringIO(''.join(lines)), skip_header=24, names=True, dtype=None, encoding=None)

Star_id_all = data_stars["Star_id"].astype(int)
Star_rad_all = data_stars["Radius"].astype(float)

# create my labels
# 0 complete disruption of both stars
# 1 merged 
# 2 unmerged 

for i in range(len(vrel_inf)):

    rp = get_rp(mini1[i], mini2[i], imp_param[i], vrel_inf[i])

    Star_id1, Star_id2 = get_star_id(coll_id[i])
    Star_rad1, Star_rad2 = get_star_radius(Star_id1, Star_id2)

    if (mfin2[i] == 0.0 and mfin1[i] != 0.0) or (mfin1[i] == 0.0 and mfin2[i] != 0.0):
        outcome = 1
    elif mfin2[i] != 0.0 and mfin1[i] != 0.0:
        outcome = 2
    elif mfin2[i] == 0.0 and mfin1[i] == 0.0:
        outcome = 0
    else:
        print(mfin1[i], mfin2[i])
        print("IDK WHAT HAPPENED HERE")

    if mini1[i] == 1.5 and mini2[i] == 0.15:
        print("YUp")
        print(coll_id[i])
        
    bs.append(imp_param[i])
    vrels.append(vrel_inf[i])
    labels.append(outcome)      
    # dms.append(dm[i])
    mass1s.append(mini1[i])
    mass2s.append(mini2[i])
    mass_ratios.append(mini1[i]/mini2[i])
    rps.append(rp / (Star_rad1 + Star_rad2)[0])
    

('Coll_id', 'Mini1', 'Mini2', 'Vrel_inf', 'ImpParam', 'Mfin1', 'Mfin2', 'Nstar', 'dM', 'dE', 'dL', 'dTheta')
velocity unit= 436.5448757818932 km/s
14187


In [9]:
data = np.array([rps, vrels,mass1s, mass_ratios ])
data_labels = np.array([labels])

np.savetxt('data.csv', data, delimiter=',', header='rp[RSUN], v_inf[km/s], Mass1[Msun], Mass_Ratio', comments='', fmt='%.6f')
np.savetxt('data_labels.csv', data_labels, delimiter=',', header='Outcome', comments='')

In [ ]:
#Append data to a file
data = np.array([bs, vrels,mass1s, mass_ratios ])
data_labels = np.array([labels])

np.savetxt('data.csv', data, delimiter=',', header='b[RSUN], v_inf[km/s], Mass1[Msun], Mass_Ratio', comments='', fmt='%.6f')
np.savetxt('data_labels.csv', data_labels, delimiter=',', header='Outcome', comments='')

In [ ]:
#Append data to a file with the dm 
# data = np.array([bs, vrels, labels, dms,])
# np.savetxt('data_dm.csv', data, delimiter=',', header='b[RSUN], v_inf[km/s], Outcome, dM ', comments='')

In [ ]:
plt.scatter(mass1s, mass2s)

In [ ]:
print((mass1s))
print((mass2s))

In [ ]:
print(np.unique(mass2s))